# Gradiente descendente para regresión lineal

**Objetivo:** encontrar la recta $\hat{y} = w\,x + b$ que mejor se ajusta a los datos.

**Analogía:** estás en una montaña con niebla y quieres llegar al valle. No ves el valle, pero sientes la pendiente bajo tus pies → das un paso cuesta abajo, vuelves a sentir, otro paso... Eso es el gradiente descendente.

| Paso | Qué hace | Fórmula |
|---|---|---|
| 1. Predecir | Modelo lineal | $\hat{y} = wx + b$ |
| 2. Medir error | Error cuadrático medio | $MSE = \frac{1}{n}\sum(\hat{y}-y)^2$ |
| 3. Gradiente | Hacia dónde **sube** el error | $\frac{\partial MSE}{\partial w} = \frac{2}{n}\sum(\hat{y}-y)\,x \quad \frac{\partial MSE}{\partial b} = \frac{2}{n}\sum(\hat{y}-y)$ |
| 4. Actualizar | Paso en sentido contrario | $w \leftarrow w - \alpha\frac{\partial MSE}{\partial w} \quad b \leftarrow b - \alpha\frac{\partial MSE}{\partial b}$ |
| 5. Repetir | Hasta que el error deje de bajar | |

$\alpha$ = *learning rate* (tamaño del paso).

In [2]:
# !pip install plotly   # descomenta si no lo tienes
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
np.random.seed(42)

## 1. Datos sintéticos
La "verdad" es $y = 3x + 4$ más ruido. El algoritmo **no** conoce estos valores; tiene que descubrirlos.

In [3]:
n = 100
x = 2 * np.random.rand(n)
y = 3 * x + 4 + np.random.randn(n) * 0.8

fig = go.Figure(go.Scatter(x=x, y=y, mode="markers", opacity=0.6, name="Datos"))
fig.update_layout(title="Datos: y = 3x + 4 + ruido", xaxis_title="x", yaxis_title="y",
                  template="plotly_white", height=400)
fig.show()

## 2. Las piezas del algoritmo

In [4]:
def predecir(x, w, b):
    """Paso 1: modelo lineal."""
    return w * x + b

def costo_mse(y, y_pred):
    """Paso 2: error cuadrático medio."""
    return np.mean((y_pred - y) ** 2)

def gradientes(x, y, y_pred):
    """Paso 3: derivadas parciales del MSE respecto a w y b."""
    error = y_pred - y
    dw = 2 * np.mean(error * x)
    db = 2 * np.mean(error)
    return dw, db

### Un solo paso, a mano
Antes del bucle, veamos qué pasa en **una** iteración partiendo de $w=0,\ b=0$.

In [6]:
w, b, alpha = 0.0, 0.0, 0.1

y_pred = predecir(x, w, b)
print(f"Costo inicial: {costo_mse(y, y_pred):.3f}")

dw, db = gradientes(x, y, y_pred)
print(f"Gradientes: dw = {dw:.3f}, db = {db:.3f}  → negativos: subir w y b reduce el error")

w -= alpha * dw
b -= alpha * db
print(f"Nuevos parámetros: w = {w:.3f}, b = {b:.3f}")
print(f"Nuevo costo: {costo_mse(y, predecir(x, w, b)):.3f}  ✅ bajó")

Costo inicial: 49.811
Gradientes: dw = -14.801, db = -13.640  → negativos: subir w y b reduce el error
Nuevos parámetros: w = 1.480, b = 1.364
Nuevo costo: 17.661  ✅ bajó


## 3. El algoritmo completo

In [8]:
def gradiente_descendente(x, y, alpha=0.1, iteraciones=200, verbose=True):
    w, b = 0.0, 0.0
    historial = []   # (w, b, costo) en cada paso

    for i in range(iteraciones):
        y_pred = predecir(x, w, b)          # 1. predecir
        costo = costo_mse(y, y_pred)        # 2. medir error
        dw, db = gradientes(x, y, y_pred)   # 3. pendiente
        w -= alpha * dw                     # 4. bajar la montaña
        b -= alpha * db
        historial.append((w, b, costo))

        if verbose and (i < 5 or i % 40 == 0 or i == iteraciones - 1):
            print(f"Iter {i:3d} | costo={costo:8.4f} | w={w:.3f}  b={b:.3f}")

    return w, b, np.array(historial)

w, b, hist = gradiente_descendente(x, y, alpha=0.1, iteraciones=200)
print(f"\nResultado:        y = {w:.3f}x + {b:.3f}")

w_ex, b_ex = np.polyfit(x, y, 1)
print(f"Solución exacta:  y = {w_ex:.3f}x + {b_ex:.3f}   (mínimos cuadrados)")

Iter   0 | costo= 49.8109 | w=1.480  b=1.364
Iter   1 | costo= 17.6608 | w=2.338  b=2.177
Iter   2 | costo=  6.5725 | w=2.831  b=2.666
Iter   3 | costo=  2.7419 | w=3.111  b=2.964
Iter   4 | costo=  1.4128 | w=3.265  b=3.150
Iter  40 | costo=  0.5333 | w=3.019  b=3.942
Iter  80 | costo=  0.5173 | w=2.867  b=4.114
Iter 120 | costo=  0.5163 | w=2.829  b=4.158
Iter 160 | costo=  0.5162 | w=2.819  b=4.168
Iter 199 | costo=  0.5162 | w=2.817  b=4.171

Resultado:        y = 2.817x + 4.171
Solución exacta:  y = 2.816x + 4.172   (mínimos cuadrados)


## 4. Cómo mejora el gradiente — visualización interactiva (Plotly)
### ▶️ Animación: la recta se ajusta iteración a iteración
Dale **Play** o mueve el slider. Arriba se ve el costo y los gradientes de cada paso.

In [9]:
xs = np.array([0, 2])
pasos = list(range(0, 30)) + list(range(30, 200, 10)) + [199]

# historial de gradientes para mostrarlos en el título
grads = [gradientes(x, y, predecir(x, 0, 0))] + \
        [gradientes(x, y, predecir(x, hw, hb)) for hw, hb, _ in hist[:-1]]

def recta(i):
    return go.Scatter(x=xs, y=predecir(xs, hist[i, 0], hist[i, 1]), mode="lines",
                      line=dict(color="crimson", width=4), name="Modelo")

def titulo(i):
    return (f"Iter {i} | w={hist[i,0]:.2f}, b={hist[i,1]:.2f} | "
            f"MSE={hist[i,2]:.3f} | ∇w={grads[i][0]:+.2f}, ∇b={grads[i][1]:+.2f}")

datos = go.Scatter(x=x, y=y, mode="markers", opacity=0.4, marker=dict(color="gray"), name="Datos")
real = go.Scatter(x=xs, y=3*xs+4, mode="lines", line=dict(dash="dash", color="green"), name="Real: 3x+4")

fig = go.Figure(
    data=[datos, real, recta(0)],
    frames=[go.Frame(data=[datos, real, recta(i)], name=str(i),
                     layout=go.Layout(title_text=titulo(i))) for i in pasos],
)
fig.update_layout(
    title=titulo(0), template="plotly_white", height=520,
    xaxis=dict(range=[0, 2], title="x"), yaxis=dict(range=[-1, 13], title="y"),
    updatemenus=[dict(type="buttons", x=0, y=-0.15, xanchor="left", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=150, redraw=True), fromcurrent=True)]),
        dict(label="⏸ Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0), mode="immediate")])])],
    sliders=[dict(currentvalue=dict(prefix="Iteración: "), y=-0.05, steps=[
        dict(label=str(i), method="animate",
             args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
        for i in pasos])],
)
fig.show()

### Curva de aprendizaje y convergencia de los parámetros
El error cae rápido al principio (gradientes grandes) y luego se estabiliza (gradientes ≈ 0).

In [ ]:
it = np.arange(len(hist))
grad_norm = [np.hypot(*g) for g in grads]

fig = make_subplots(rows=1, cols=3, subplot_titles=("MSE (log)", "Tamaño del gradiente |∇| (log)", "Parámetros"))
fig.add_trace(go.Scatter(x=it, y=hist[:, 2], name="MSE", line=dict(color="crimson")), 1, 1)
fig.add_trace(go.Scatter(x=it, y=grad_norm, name="|∇|", line=dict(color="purple")), 1, 2)
fig.add_trace(go.Scatter(x=it, y=hist[:, 0], name="w"), 1, 3)
fig.add_trace(go.Scatter(x=it, y=hist[:, 1], name="b"), 1, 3)
fig.add_hline(y=w_ex, line_dash="dot", row=1, col=3)
fig.add_hline(y=b_ex, line_dash="dot", row=1, col=3)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_xaxes(title_text="Iteración")
fig.update_layout(template="plotly_white", height=400, hovermode="x unified")
fig.show()

### 🏔️ La montaña del error en 3D
Cada punto $(w, b)$ tiene un error. El algoritmo baja por la pendiente hasta el valle. **Gira la gráfica con el dedo o el mouse.**

In [ ]:
W, B = np.meshgrid(np.linspace(-1, 6, 80), np.linspace(-1, 7, 80))
Z = np.mean((W[..., None] * x + B[..., None] - y) ** 2, axis=-1)

fig = go.Figure()
fig.add_trace(go.Surface(x=W, y=B, z=Z, colorscale="Viridis", opacity=0.75, showscale=False))
fig.add_trace(go.Scatter3d(x=np.r_[0, hist[:, 0]], y=np.r_[0, hist[:, 1]],
                           z=np.r_[costo_mse(y, predecir(x, 0, 0)), hist[:, 2]],
                           mode="lines+markers", marker=dict(size=3, color="red"),
                           line=dict(color="red", width=5), name="Camino del gradiente"))
fig.add_trace(go.Scatter3d(x=[w], y=[b], z=[hist[-1, 2]], mode="markers",
                           marker=dict(size=8, color="gold", symbol="diamond"), name="Mínimo"))
fig.update_layout(scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="MSE"),
                  height=600, title="Superficie de error y trayectoria")
fig.show()

### Vista desde arriba (curvas de nivel)
Las flechas muestran que cada paso va **perpendicular** a las curvas de nivel: esa es la dirección de máximo descenso.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=np.log10(Z), colorscale="RdBu_r",
                         contours=dict(showlabels=False), showscale=False))
fig.add_trace(go.Scatter(x=np.r_[0, hist[:40, 0]], y=np.r_[0, hist[:40, 1]],
                         mode="lines+markers", marker=dict(color="black", size=6),
                         line=dict(color="black"), name="Primeros 40 pasos"))
fig.add_trace(go.Scatter(x=[w], y=[b], mode="markers", marker=dict(symbol="star", size=18, color="gold"), name="Mínimo"))
fig.update_layout(xaxis_title="w (pendiente)", yaxis_title="b (intercepto)",
                  template="plotly_white", height=500)
fig.show()

## 5. El learning rate importa
- **Muy pequeño:** llega, pero lentísimo.
- **Adecuado:** converge rápido.
- **Muy grande:** se pasa del mínimo y **diverge**.

Haz clic en la leyenda para ocultar/mostrar curvas.

In [ ]:
tasas = {0.01: "muy pequeño", 0.05: "pequeño", 0.1: "adecuado", 0.5: "ya diverge", 0.6: "muy grande"}
fig = go.Figure()
for a, desc in tasas.items():
    _, _, h = gradiente_descendente(x, y, alpha=a, iteraciones=50, verbose=False)
    fig.add_trace(go.Scatter(y=h[:, 2], mode="lines", name=f"α={a} ({desc})"))
    print(f"α={a:<5} {desc:<12} → costo final: {h[-1, 2]:.4g}")
fig.update_layout(yaxis_type="log", xaxis_title="Iteración", yaxis_title="MSE",
                  title="Efecto del learning rate", template="plotly_white", height=450)
fig.show()

α=0.01  muy pequeño  → costo final: 1.496
α=0.05  pequeño      → costo final: 0.5668
α=0.1   adecuado     → costo final: 0.5254
α=0.5   ya diverge   → costo final: 2.354e+04
α=0.6   muy grande   → costo final: 2.087e+18


## Resumen
1. Empezamos con parámetros al azar ($w=0, b=0$).
2. Calculamos el error y su **gradiente** (la pendiente).
3. Damos un paso **en contra** del gradiente, de tamaño $\alpha$.
4. Repetimos → los parámetros convergen a $w \approx 3,\ b \approx 4$.

**Para experimentar:** cambia `alpha`, `iteraciones`, el ruido de los datos o el punto de inicio y observa qué pasa.

---
# Parte 2: Gradiente descendente para regresión logística

Ahora el objetivo no es predecir un número, sino una **clase**: $y \in \{0, 1\}$ (¿spam o no?, ¿enfermo o sano?).

La receta del gradiente descendente es **la misma**; solo cambian dos piezas:

| Paso | Regresión lineal | Regresión logística |
|---|---|---|
| 1. Predecir | $\hat{y} = wx + b$ | $p = \sigma(w \cdot x + b)$, una **probabilidad** |
| 2. Medir error | MSE | **Entropía cruzada** (log-loss) |
| 3. Gradiente | $\frac{2}{n}\sum(\hat{y}-y)\,x$ | $\frac{1}{n}\sum(p-y)\,x$ |
| 4. Actualizar | $w \leftarrow w - \alpha\,\nabla$ | igual |

## 6. El modelo: la función sigmoide

Una recta $z = w \cdot x + b$ da valores en $(-\infty, \infty)$, pero una probabilidad debe estar en $[0, 1]$. La **sigmoide** "aplasta" cualquier número a ese rango:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- $z \gg 0 \Rightarrow \sigma(z) \approx 1$ → clase 1
- $z \ll 0 \Rightarrow \sigma(z) \approx 0$ → clase 0
- $z = 0 \Rightarrow \sigma(z) = 0.5$ → **frontera de decisión**

In [ ]:
def sigmoide(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 200)
fig = go.Figure(go.Scatter(x=z, y=sigmoide(z), mode="lines", line=dict(width=4, color="royalblue"), name="σ(z)"))
fig.add_hline(y=0.5, line_dash="dot", annotation_text="p = 0.5 (frontera)")
fig.add_vline(x=0, line_dash="dot")
fig.update_layout(title="Sigmoide: convierte cualquier número en una probabilidad",
                  xaxis_title="z = w·x + b", yaxis_title="p = σ(z)", template="plotly_white", height=400)
fig.show()

## 7. ¿Cómo medimos el error? La entropía

### 7.1 Intuición: la "sorpresa"
En teoría de la información, la **sorpresa** de que ocurra un evento con probabilidad $p$ es

$$\text{sorpresa} = -\log(p)$$

- Si estabas seguro ($p = 1$) y ocurre → sorpresa $= 0$.
- Si decías que era casi imposible ($p = 0.01$) y ocurre → sorpresa enorme ($\approx 4.6$).

La **entropía** es la sorpresa *promedio* de una distribución: $H(p) = -\sum_k p_k \log p_k$. Mide cuánta incertidumbre hay. Una moneda justa ($p=0.5$) tiene la máxima entropía; una moneda trucada que siempre cae cara tiene entropía 0.

### 7.2 Entropía cruzada binaria (*log-loss*)
Para clasificar, medimos la sorpresa del modelo frente a la **etiqueta real**. Si el modelo dice $p = P(y=1)$:

$$L(y, p) = -\big[\,y \log(p) + (1 - y)\log(1 - p)\,\big]$$

Fíjate que solo uno de los dos términos "se enciende":

| Etiqueta real | Pérdida | Interpretación |
|---|---|---|
| $y = 1$ | $-\log(p)$ | castiga cuando $p$ es **pequeña** |
| $y = 0$ | $-\log(1-p)$ | castiga cuando $p$ es **grande** |

El costo total es el promedio sobre los $n$ datos: $J(w,b) = \frac{1}{n}\sum_i L(y_i, p_i)$.

In [ ]:
p = np.linspace(0.001, 0.999, 500)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Pérdida según la etiqueta real", "Entropía de una moneda H(p)"))
fig.add_trace(go.Scatter(x=p, y=-np.log(p), name="y = 1 →  −log(p)",
                         line=dict(color="seagreen", width=4)), 1, 1)
fig.add_trace(go.Scatter(x=p, y=-np.log(1 - p), name="y = 0 →  −log(1−p)",
                         line=dict(color="crimson", width=4)), 1, 1)

# algunos puntos de referencia para y = 1
for pi in [0.99, 0.9, 0.5, 0.1, 0.01]:
    fig.add_trace(go.Scatter(x=[pi], y=[-np.log(pi)], mode="markers+text", text=[f"{-np.log(pi):.2f}"],
                             textposition="top right", marker=dict(size=9, color="seagreen"),
                             showlegend=False), 1, 1)

H = -(p * np.log(p) + (1 - p) * np.log(1 - p))
fig.add_trace(go.Scatter(x=p, y=H, name="H(p)", line=dict(color="purple", width=4)), 1, 2)

fig.update_xaxes(title_text="p = probabilidad predicha de la clase 1", range=[-0.02, 1.1], row=1, col=1)
fig.update_xaxes(title_text="p = P(cara)", row=1, col=2)
fig.update_yaxes(title_text="Pérdida", range=[0, 5], row=1, col=1)
fig.update_yaxes(title_text="Entropía", row=1, col=2)
fig.update_layout(template="plotly_white", height=450)
fig.show()

print("Si y = 1:")
for pi in [0.99, 0.9, 0.5, 0.1, 0.01]:
    print(f"   el modelo dice p = {pi:<5} → pérdida = {-np.log(pi):.3f}")

**Lectura del gráfico:**
- Curva verde ($y=1$): si el modelo acierta con confianza ($p \to 1$) la pérdida es $\approx 0$; si se equivoca con confianza ($p \to 0$) la pérdida **explota** hacia $\infty$.
- Curva roja ($y=0$): es el espejo; castiga las $p$ cercanas a 1.
- Equivocarse con seguridad cuesta mucho más que dudar ($p = 0.5$ cuesta $\log 2 \approx 0.69$).

> **¿Por qué no usar MSE?** Con la sigmoide, el MSE deja de ser una "montaña con un solo valle" (no es convexo) y además da gradientes minúsculos cuando el modelo se equivoca mucho. La entropía cruzada **sí es convexa** en $(w, b)$ → el gradiente descendente siempre llega al mínimo global.

## 8. Derivando el gradiente

Para un solo dato, la cadena de cálculo es

$$x \;\xrightarrow{\;z = w\cdot x + b\;}\; z \;\xrightarrow{\;\sigma\;}\; p \;\xrightarrow{\;L\;}\; L(y, p)$$

Usamos la **regla de la cadena**: $\dfrac{\partial L}{\partial w_j} = \dfrac{\partial L}{\partial p}\cdot\dfrac{\partial p}{\partial z}\cdot\dfrac{\partial z}{\partial w_j}$

**① Derivada de la pérdida respecto a $p$:**
$$\frac{\partial L}{\partial p} = -\frac{y}{p} + \frac{1-y}{1-p} = \frac{p - y}{p(1-p)}$$

**② Derivada de la sigmoide** (tiene una forma muy elegante):
$$\sigma(z) = (1+e^{-z})^{-1} \;\Rightarrow\; \frac{\partial p}{\partial z} = \frac{e^{-z}}{(1+e^{-z})^2} = \underbrace{\frac{1}{1+e^{-z}}}_{p}\cdot\underbrace{\frac{e^{-z}}{1+e^{-z}}}_{1-p} = p(1-p)$$

**③ Derivada de la parte lineal:**
$$\frac{\partial z}{\partial w_j} = x_j \qquad \frac{\partial z}{\partial b} = 1$$

**Juntando ① · ②:** ¡el $p(1-p)$ se cancela!
$$\frac{\partial L}{\partial z} = \frac{p - y}{p(1-p)}\cdot p(1-p) = p - y$$

**Resultado final**, promediando sobre los $n$ datos:

$$\boxed{\;\frac{\partial J}{\partial w_j} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)\,x_{ij} \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)\;}$$

En forma matricial: $\nabla_w J = \frac{1}{n} X^\top (p - y)$.

💡 **Es casi idéntico al de la regresión lineal**: "error × entrada". La diferencia es que el error ahora es $p - y$ (probabilidad menos etiqueta).

### Comprobación numérica del gradiente
Verifiquemos la fórmula comparándola con una derivada aproximada: $\frac{\partial J}{\partial w} \approx \frac{J(w+\varepsilon) - J(w-\varepsilon)}{2\varepsilon}$.

In [ ]:
# un mini ejemplo 1D para comprobar la derivación
x_chk = np.array([-2.0, -1.0, 0.5, 1.0, 2.5])
y_chk = np.array([0, 0, 1, 0, 1])
w0, b0, eps = 0.7, -0.2, 1e-6

def J(w, b):
    p = sigmoide(w * x_chk + b)
    return -np.mean(y_chk * np.log(p) + (1 - y_chk) * np.log(1 - p))

p0 = sigmoide(w0 * x_chk + b0)
dw_formula = np.mean((p0 - y_chk) * x_chk)
db_formula = np.mean(p0 - y_chk)
dw_numerico = (J(w0 + eps, b0) - J(w0 - eps, b0)) / (2 * eps)
db_numerico = (J(w0, b0 + eps) - J(w0, b0 - eps)) / (2 * eps)

print(f"dJ/dw  fórmula = {dw_formula:.6f}   numérico = {dw_numerico:.6f}")
print(f"dJ/db  fórmula = {db_formula:.6f}   numérico = {db_numerico:.6f}   ✅ coinciden")

## 9. El cálculo

### 9.1 Datos sintéticos
Dos grupos de puntos en el plano $(x_1, x_2)$ que se solapan un poco. El modelo debe aprender $w_1, w_2, b$ para separarlos.

In [ ]:
np.random.seed(0)
n_c = 100
X0 = np.random.randn(n_c, 2) * 1.0 + [-1.0, -1.0]   # clase 0
X1 = np.random.randn(n_c, 2) * 1.0 + [1.5, 1.0]     # clase 1
X = np.vstack([X0, X1])
yc = np.r_[np.zeros(n_c), np.ones(n_c)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=X0[:, 0], y=X0[:, 1], mode="markers", marker=dict(color="crimson"), name="y = 0"))
fig.add_trace(go.Scatter(x=X1[:, 0], y=X1[:, 1], mode="markers", marker=dict(color="seagreen"), name="y = 1"))
fig.update_layout(title="Datos de clasificación", xaxis_title="x₁", yaxis_title="x₂",
                  template="plotly_white", height=450)
fig.show()

### 9.2 Las piezas del algoritmo

In [ ]:
def predecir_proba(X, w, b):
    """Paso 1: probabilidad de la clase 1."""
    return sigmoide(X @ w + b)

def costo_entropia(y, p, eps=1e-12):
    """Paso 2: entropía cruzada binaria promedio."""
    p = np.clip(p, eps, 1 - eps)          # evita log(0)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradientes_log(X, y, p):
    """Paso 3: derivadas parciales de la entropía cruzada."""
    error = p - y
    dw = X.T @ error / len(y)             # vector (dw1, dw2)
    db = np.mean(error)
    return dw, db

def exactitud(y, p):
    return np.mean((p >= 0.5) == y)

### 9.3 Un solo paso, a mano
Partimos de $w = (0, 0),\ b = 0$: el modelo dice $p = 0.5$ para **todos** los puntos (no sabe nada). El costo inicial es $\log 2 \approx 0.693$.

In [ ]:
w_l, b_l, alpha = np.zeros(2), 0.0, 0.5

p = predecir_proba(X, w_l, b_l)
print(f"Costo inicial: {costo_entropia(yc, p):.4f}  (log 2 = {np.log(2):.4f})")
print(f"Exactitud inicial: {exactitud(yc, p):.0%}")

dw_l, db_l = gradientes_log(X, yc, p)
print(f"Gradientes: dw = {np.round(dw_l, 3)}, db = {db_l:.3f}")

w_l = w_l - alpha * dw_l
b_l = b_l - alpha * db_l
p = predecir_proba(X, w_l, b_l)
print(f"Nuevos parámetros: w = {np.round(w_l, 3)}, b = {b_l:.3f}")
print(f"Nuevo costo: {costo_entropia(yc, p):.4f}  ✅ bajó  |  exactitud: {exactitud(yc, p):.0%}")

### 9.4 El algoritmo completo

In [ ]:
def gradiente_descendente_logistico(X, y, alpha=0.5, iteraciones=300, verbose=True):
    w, b = np.zeros(X.shape[1]), 0.0
    historial = []   # (w1, w2, b, costo, exactitud) ANTES de cada actualización

    for i in range(iteraciones):
        p = predecir_proba(X, w, b)           # 1. predecir
        costo = costo_entropia(y, p)          # 2. medir error
        dw, db = gradientes_log(X, y, p)      # 3. pendiente
        historial.append((*w, b, costo, exactitud(y, p)))
        w = w - alpha * dw                    # 4. bajar la montaña
        b = b - alpha * db

        if verbose and (i < 5 or i % 50 == 0 or i == iteraciones - 1):
            print(f"Iter {i:3d} | costo={costo:.4f} | exactitud={historial[-1][-1]:.0%} | "
                  f"w=({w[0]:.3f}, {w[1]:.3f})  b={b:.3f}")

    p = predecir_proba(X, w, b)
    historial.append((*w, b, costo_entropia(y, p), exactitud(y, p)))
    return w, b, np.array(historial)

w_log, b_log, hist_log = gradiente_descendente_logistico(X, yc, alpha=0.5, iteraciones=300)
print(f"\nResultado: p = σ({w_log[0]:.3f}·x₁ + {w_log[1]:.3f}·x₂ {b_log:+.3f})")

# comparación con scikit-learn (C enorme = prácticamente sin regularización)
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1e10, max_iter=10_000).fit(X, yc)
print(f"sklearn:   p = σ({sk.coef_[0][0]:.3f}·x₁ + {sk.coef_[0][1]:.3f}·x₂ {sk.intercept_[0]:+.3f})")

## 10. Visualización: la clasificación mientras se minimiza el error

### ▶️ Animación: la frontera de decisión se acomoda iteración a iteración
- El **fondo** es la probabilidad $p(x)$ que predice el modelo (rojo → clase 0, verde → clase 1).
- La **línea negra** es la frontera de decisión $p = 0.5$, es decir $w_1x_1 + w_2x_2 + b = 0$.
- Los puntos marcados con **✕** están mal clasificados.

Dale **Play** o mueve el slider y observa cómo baja la entropía cruzada en el título.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-4.5, 5, 60), np.linspace(-4.5, 4.5, 60))
grid = np.c_[g1.ravel(), g2.ravel()]
pasos_log = list(range(0, 15)) + list(range(15, 100, 10)) + list(range(100, 301, 25))
pasos_log = sorted(set(min(i, len(hist_log) - 1) for i in pasos_log))
escala_rg = [[0, "rgb(214,39,40)"], [0.5, "rgb(245,245,245)"], [1, "rgb(44,160,44)"]]

def trazas(i):
    w_i, b_i = hist_log[i, :2], hist_log[i, 2]
    P = predecir_proba(grid, w_i, b_i).reshape(g1.shape)
    pred = predecir_proba(X, w_i, b_i) >= 0.5
    mal = pred != yc
    fondo = go.Contour(x=g1[0], y=g2[:, 0], z=P, zmin=0, zmax=1, colorscale=escala_rg, opacity=0.55,
                       contours=dict(start=0, end=1, size=0.1), line=dict(width=0),
                       colorbar=dict(title="p(y=1)"), name="p(x)")
    frontera = go.Contour(x=g1[0], y=g2[:, 0], z=P, showscale=False, contours_coloring="lines",
                          contours=dict(start=0.5, end=0.5, size=1), line=dict(color="black", width=4),
                          colorscale=[[0, "black"], [1, "black"]], name="Frontera p = 0.5")
    puntos = []
    for clase, color in [(0, "crimson"), (1, "seagreen")]:
        for es_mal, simb in [(False, "circle"), (True, "x")]:
            m = (yc == clase) & (mal == es_mal)
            puntos.append(go.Scatter(x=X[m, 0], y=X[m, 1], mode="markers",
                                     marker=dict(color=color, symbol=simb, size=11 if es_mal else 8,
                                                 line=dict(color="black", width=1)),
                                     name=f"y={clase} {'mal' if es_mal else 'bien'} clasificado"))
    return [fondo, frontera] + puntos

def titulo_log(i):
    w1, w2, b_, J_, acc = hist_log[i]
    return (f"Iter {i} | w=({w1:.2f}, {w2:.2f}), b={b_:.2f} | "
            f"Entropía cruzada={J_:.3f} | Exactitud={acc:.0%}")

fig = go.Figure(
    data=trazas(0),
    frames=[go.Frame(data=trazas(i), name=str(i), layout=go.Layout(title_text=titulo_log(i)))
            for i in pasos_log],
)
fig.update_layout(
    title=titulo_log(0), template="plotly_white", height=600,
    xaxis=dict(range=[-4.5, 5], title="x₁"), yaxis=dict(range=[-4.5, 4.5], title="x₂"),
    legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0),
    updatemenus=[dict(type="buttons", x=0, y=-0.15, xanchor="left", buttons=[
        dict(label="▶ Play", method="animate",
             args=[None, dict(frame=dict(duration=150, redraw=True), fromcurrent=True)]),
        dict(label="⏸ Pausa", method="animate",
             args=[[None], dict(frame=dict(duration=0), mode="immediate")])])],
    sliders=[dict(currentvalue=dict(prefix="Iteración: "), y=-0.05, steps=[
        dict(label=str(i), method="animate",
             args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
        for i in pasos_log])],
)
fig.show()

### Curva de aprendizaje: el error baja y la exactitud sube
La entropía cruzada arranca en $\log 2 \approx 0.693$ (el modelo "adivina" 50/50) y desciende hasta estabilizarse. Los parámetros convergen a la solución de scikit-learn (líneas punteadas).

In [ ]:
it_l = np.arange(len(hist_log))
fig = make_subplots(rows=1, cols=3, subplot_titles=("Entropía cruzada", "Exactitud", "Parámetros"))
fig.add_trace(go.Scatter(x=it_l, y=hist_log[:, 3], name="Entropía cruzada", line=dict(color="crimson")), 1, 1)
fig.add_hline(y=np.log(2), line_dash="dot", annotation_text="log 2", row=1, col=1)
fig.add_trace(go.Scatter(x=it_l, y=hist_log[:, 4], name="Exactitud", line=dict(color="seagreen")), 1, 2)
for k, nombre, ref in [(0, "w₁", sk.coef_[0][0]), (1, "w₂", sk.coef_[0][1]), (2, "b", sk.intercept_[0])]:
    fig.add_trace(go.Scatter(x=it_l, y=hist_log[:, k], name=nombre), 1, 3)
    fig.add_hline(y=ref, line_dash="dot", row=1, col=3)
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.update_xaxes(title_text="Iteración")
fig.update_layout(template="plotly_white", height=400, hovermode="x unified")
fig.show()

### 🏔️ La montaña de la entropía cruzada en 3D
Fijamos $b$ en su valor final y dibujamos el costo para cada par $(w_1, w_2)$. A diferencia del MSE con sigmoide, esta superficie es **convexa**: un solo valle. La línea roja es la trayectoria del gradiente (proyectada en ese corte).

In [ ]:
W1, W2 = np.meshgrid(np.linspace(-1, 4, 70), np.linspace(-1, 4, 70))
Zc = np.array([[costo_entropia(yc, predecir_proba(X, np.array([a, c]), b_log))
                for a, c in zip(fila_a, fila_c)] for fila_a, fila_c in zip(W1, W2)])
J_tray = [costo_entropia(yc, predecir_proba(X, h[:2], b_log)) for h in hist_log]

fig = go.Figure()
fig.add_trace(go.Surface(x=W1, y=W2, z=Zc, colorscale="Viridis", opacity=0.8, showscale=False))
fig.add_trace(go.Scatter3d(x=hist_log[:, 0], y=hist_log[:, 1], z=J_tray, mode="lines+markers",
                           marker=dict(size=3, color="red"), line=dict(color="red", width=5),
                           name="Camino del gradiente"))
fig.add_trace(go.Scatter3d(x=[w_log[0]], y=[w_log[1]], z=[J_tray[-1]], mode="markers",
                           marker=dict(size=8, color="gold", symbol="diamond"), name="Mínimo"))
fig.update_layout(scene=dict(xaxis_title="w₁", yaxis_title="w₂", zaxis_title="Entropía cruzada"),
                  height=600, title=f"Superficie de error (b fijo = {b_log:.2f})")
fig.show()

## Resumen — regresión logística
1. El modelo predice una **probabilidad** con la sigmoide: $p = \sigma(w \cdot x + b)$.
2. El error se mide con la **entropía cruzada**: castiga mucho equivocarse con confianza ($-\log p$ si $y=1$, $-\log(1-p)$ si $y=0$).
3. Por la regla de la cadena, el gradiente queda $\nabla_w J = \frac{1}{n}X^\top(p - y)$: "error × entrada", igual que en regresión lineal.
4. Repetimos el paso $w \leftarrow w - \alpha \nabla$ → la frontera de decisión gira y se desplaza hasta separar las clases lo mejor posible.

**Para experimentar:** acerca los centros de las dos clases (más solapamiento), cambia `alpha` o agrega un dato atípico y observa cómo responde la frontera.